# standard codes

In [18]:
import numpy as np
import requests
import os
import time
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')
import json
import glob
from IPython.display import clear_output
from scipy import stats
from scipy.stats import t

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
tng_api_key = userdata.get('TNG_API_KEY')
baseUrl = 'http://www.tng-project.org/api/'
headers = {"api-key":tng_api_key}

In [11]:
def get(path, params=None, out_filename=None):
    headers = {"api-key":tng_api_key}
    r = requests.get(path, params=params, headers=headers)
    r.raise_for_status()

    if out_filename is not None:
        with open(out_filename, 'wb') as f:
            f.write(r.content)
        return out_filename

    if r.headers['content-type'] == 'application/json':
        return r.json()

    if 'content-disposition' in r.headers:
        filename = r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        return filename

    return r

In [12]:
r = get(baseUrl)

for simulation in r['simulations']: #only get TNG50
    if simulation['name'] == 'TNG50-1':
        url = simulation['url']
        break

tng50 = get(url)

url = 'http://www.tng-project.org/api/TNG50-1/snapshots/z=1.8/'
snapshot = get(url)

#new codes

In [13]:
drive_path = '/content/drive/MyDrive/docs'
os.makedirs(drive_path, exist_ok=True)

In [15]:
files = np.sort(glob.glob(f"{drive_path}/sub*.json"))
logs = []
max_lines = 15

def log_message(message):
    logs.append(message)
    if len(logs) > max_lines:
        logs.pop(0) # Remove the oldest message
    clear_output(wait=True)
    for msg in logs:
        print(msg)

for i in range(snapshot['num_groups_subfind']):
    if i % int(snapshot['num_groups_subfind']/1000) == 0:
        log_message(str(i))

    if f"{drive_path}/sub{i}.json" in files:
        log_message(f"skipping i: {i}") # Log skipping
        continue # Skip the download part if file exists

    sub_url = f"http://www.tng-project.org/api/TNG50-1/snapshots/{snapshot['number']}/subhalos/{i}/"
    subhalo = get((sub_url), out_filename=f"{drive_path}/sub{i}.json")

    if i > 6:
        break

skipping i: 31538
skipping i: 31539
skipping i: 31540
skipping i: 31541
skipping i: 31542
skipping i: 31543
skipping i: 31544
skipping i: 31545
skipping i: 31546
skipping i: 31547
skipping i: 31548
skipping i: 31549
skipping i: 31550
skipping i: 31551
skipping i: 31552


KeyboardInterrupt: 

In [16]:
FIRST_FEW = 10
bhmdot = np.zeros(FIRST_FEW) -1
sfr = np.zeros(FIRST_FEW) -1

print('bhmdot: ',bhmdot)
print("sfr: ",sfr)

save_file = f"{drive_path}/bh_sfr.npz"
if os.path.exists(save_file):
    data = np.load(save_file)
    bhmdot[:FIRST_FEW] = data['bhmdot'][:FIRST_FEW]
    sfr[:FIRST_FEW] = data['sfr'][:FIRST_FEW]

print('bhmdot: ',bhmdot)
print("sfr: ",sfr)

for i in range(FIRST_FEW):
    if bhmdot[i] >= 0:
        continue
    if sfr[i] >= 0:
        continue
    with open(files[i], 'r') as fid:
        data = json.load(fid)
    bhmdot[i] = data['bhmdot']
    sfr[i] = data['sfr']
    if i > FIRST_FEW:
        break
    np.savez_compressed(save_file, bhmdot=bhmdot, sfr=sfr)

bhmdot:  [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
sfr:  [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
bhmdot:  [0.00351652 0.0166792  0.00051153 0.         0.         0.
 0.         0.         0.         0.        ]
sfr:  [410.991    61.4731    1.57871   0.        0.        0.        0.
   0.        0.        0.     ]


In [17]:
t_stat, p_val = stats.ttest_ind(bhmdot, sfr)

print(f"t_stat: {t_stat: .2f}")
print(f"p-val: {p_val: .3f}")

alpha = 0.5
if p_val < alpha:
    print(f"Since p-value ({p_val: .3f} is < than alpha ({alpha})), we reject the null hypothesis")
elif p_val > alpha:
    print(f"Since p-value ({p_val: .3f} is > than alpha ({alpha})), we accept the null hypothesis")

t_stat: -1.16
p-val:  0.261
Since p-value ( 0.261 is < than alpha (0.5)), we reject the null hypothesis


In [19]:
def manual_ttest_ind(bhmdot, sfr):
    n1, n2= len(bhmdot), len(sfr)
    mean1, mean2 = np.mean(bhmdot), np.mean(sfr)
    var1, var2 = np.var(bhmdot, ddof=1), np.var(sfr, ddof=1)

    se_diff_sq = (var1/n1 + var2/n2)
    se_diff =np.sqrt(se_diff_sq)
    t_stat = (mean1 - mean2)/se_diff

    df_num = se_diff_sq**2
    df_den = ((var1**2/ n1**2)/(n1-1)) + ((var2**2/ n2**2)/(n2-1))
    df = df_num/df_den

    p_value = 2 * t.sf(np.abs(t_stat),df)
    return t_stat, p_value

manual_t_stat, manual_p_val = manual_ttest_ind(bhmdot, sfr)
print(f'Manual t-statistic: {manual_t_stat: .2f}')
print(f'Manual p-value: {manual_p_val: .3f}')

Manual t-statistic: -1.16
Manual p-value:  0.276


In [21]:
np.corrcoef(bhmdot, sfr)

array([[1.        , 0.24410407],
       [0.24410407, 1.        ]])